Generate Collect Earth Online plots for a SIDS country, following the CEO example plot GeoJSON.
Each plot is one cell of the 30m grid for the country's region from `get_gridspec`, so plots are exactly the GeoMAD and
LULC pixels. Plots are stratified by ESA WorldCover loaded onto that grid and mapped to our level 1 classes.

1. Load GADM for the country and buffer 200m (antimeridian fixed).
2. Load ESA WorldCover (level 1 classes) onto each 30m grid tile covering the buffered country and merge the tiles.
3. Choose the plots with `random_sampling` (equal stratified random). Classes with too few pixels give all they have,
   so the total can be a little under N_PLOTS.
4. Write a zipped shapefile in EPSG:4326 with a square polygon per plot, with PLOTID and ExtraInfo (the country name).
   The class is not written, so it does not bias the interpreters.

In [4]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import xarray as xr
from odc.geo.geom import Geometry

from ldn.grids import get_gridspec
from ldn.random_sampling import random_sampling
from ldn.training_data import LULC_PRODUCTS, get_buffered_country, load_and_prepare, product_nodata_value
from ldn.utils import DEP_COUNTRIES_AND_CODES, TRAINING_DATA_YEAR, WGS84, get_analysis_epsg

In [ ]:
# country_of_interest = {"Fiji": "FJI"} # TODO: Test with antimeridian
country_of_interest = {"Anguilla": "AIA"}
BUFFER_M = 200  # For buffering country polygons
N_PLOTS = 3_000
SEED = 0
WORLDCOVER = LULC_PRODUCTS[0]
assert WORLDCOVER["product"] == "esa-worldcover", "Product must be `esa-worldcover`"  # Safeguard

name = next(iter(country_of_interest))
PLOTS_PATH = Path(f"country_plots/{name}_plots.shp.zip")

In [ ]:
region = "pacific" if name in DEP_COUNTRIES_AND_CODES else "non-pacific"  # TODO: Make more robust?
analysis_crs = get_analysis_epsg(region)
grid = get_gridspec(region)
country_wgs84_buffered = get_buffered_country(country_of_interest, analysis_crs, BUFFER_M).dissolve()
# TODO: Does this just take the first geometry? Does this work for multipolygons?
geom_country_wgs84_buffered = Geometry(country_wgs84_buffered.to_crs(analysis_crs).geometry.iloc[0], crs=analysis_crs)

# Per tile for a country, load WC (clipped to country buffered 200m)
tiles = [
    load_and_prepare(geobox, country_wgs84_buffered, WORLDCOVER, TRAINING_DATA_YEAR)
    for _, geobox in grid.tiles_from_geopolygon(geom_country_wgs84_buffered)
]
bands = [tile[WORLDCOVER["output_band"]] for tile in tiles]

# TODO: Make samples per tile and then merge these?
# TODO: Kiribati covers a huge area (but is very sparse). Could hit RAM limit.
# TODO: Fiji could be an issue with the current approach - belting the world blowing up memory.
# TODO: Tiles would then have their own stratification, but that's ok.
# TODO: How many samples to try for in each tile?
# Fiji is 26 tiles.
# Singapore is 2.
# Anguilla is 4 even though it would easily fit into 1.
# Guyana is 37.
# Belize is 9
# Cabo Verde is 10.
# The approach is fair enough because the amount of plots/samples scales roughly with country area.
# So for Fiji's 26 tiles if each one makes 300 stratified equal (43 per class) samples per tile, then nationally it
#  will have 7800 (1115 per class).
# The problem is that each tile does not have an even amount of the country's land. I could calculate proportion of
#  land in that tile, and then allocate that proportion of 3000 samples to be created.
x = np.unique(np.concatenate([band.x.values for band in bands]))
y = np.unique(np.concatenate([band.y.values for band in bands]))[::-1]
wc = xr.DataArray(
    np.full((len(y), len(x)), product_nodata_value, dtype="uint8"), coords={"y": y, "x": x}, dims=("y", "x")
)
for band in bands:
    wc.loc[{"y": band.y, "x": band.x}] = band.values
wc = wc.rio.write_crs(analysis_crs)
print(f"{name}: merged {len(tiles)} tiles into {wc.shape}")

Anguilla: merged 4 tiles into (1755, 1626)


In [ ]:
samples = random_sampling(
    da=wc.rename({"y": "latitude", "x": "longitude"}),
    n=N_PLOTS,
    min_sample_n=None,  # Not used by equal stratified random. TODO: make this param optional
    sampling="equal_stratified_random",
    drop_value=product_nodata_value,
    seed=SEED,
)
print(samples["lulc"].value_counts().sort_index())

# Shuffle so PLOTID order does not follow the class
samples = samples.sample(frac=1, random_state=SEED)
centres = samples.geometry.to_crs(analysis_crs)
plots = gpd.GeoDataFrame(
    {"PLOTID": np.arange(1, len(samples) + 1).astype(str), "ExtraInfo": name},
    geometry=centres.buffer(grid.resolution.x / 2, cap_style="square").to_crs(WGS84).values,
)
plots.to_file(PLOTS_PATH, driver="ESRI Shapefile")
print(f"Wrote {len(plots)} plots to {PLOTS_PATH}")

lulc
1    429
2    429
3    408
4    429
5    429
6    429
7    429
Name: count, dtype: int64
Wrote 2982 plots to country_plots/Anguilla_plots.shp.zip
